In [1]:
# SEED mengunci undian acak di notebook ini.
SEED = 42


# Modul 3b: Regularisasi (Ridge dan Lasso)

Cerita lengkapnya ada di `README.md`. Hitung dulu, baru jalankan kode.

Petunjuk adalah angka yang dibaca untuk menyusun tebakan.
Koefisien adalah angka pengali pada satu petunjuk di garis tebakan.
Penalti adalah tambahan yang membesar bila koefisien membesar.
Simpangan baku mengukur sebaran angka di sekitar rata-ratanya.
Regresi linear menebak jawaban angka sebagai jumlah petunjuk yang masing-masing dikalikan koefisien, ditambah satu bilangan geser.
intercept_ adalah nama tempat scikit-learn menyimpan bilangan geser.

coef_ adalah nama tempat scikit-learn menyimpan koefisien.
Fungsi kerugian adalah angka kesalahan yang diperkecil saat model belajar; pada regresi linear tanpa penalti, angka itu adalah jumlah kuadrat selisih antara tebakan dan jawaban.
Data latih dan data uji berarti data dibagi dua: data latih dipakai model untuk belajar; data uji disisihkan dan tidak dilihat model saat belajar, dipakai untuk mengukur seberapa baik model pada data baru.
fit adalah langkah saat model belajar dari data yang diberikan kepadanya.
Overfitting berarti model terlalu menyesuaikan diri dengan data latih, termasuk kebetulan-kebetulannya, sehingga hasilnya bagus di data latih tetapi buruk di data baru.
Baseline adalah cara menebak paling sederhana (misalnya selalu menebak rata-rata atau kelas yang paling banyak) yang dipakai sebagai pembanding; model yang baik harus mengalahkannya.
Rata-rata selisih mutlak adalah rata-rata jarak antara tebakan dan jawaban, jarak berarti selisih yang tanda minusnya dibuang, jumlah jarak dibagi banyak baris yang diukur, semakin kecil semakin dekat, nilai terbaik adalah nol, dan fungsi scikit-learn untuk hitungan yang sama bernama mean_absolute_error.
alpha adalah angka yang mengalikan penalti; makin besar nilainya, makin kuat penalti itu.
StandardScaler menggeser tiap petunjuk supaya rata-ratanya nol dan simpangan bakunya satu, memakai rata-rata dan simpangan baku data yang dipelajari saat fit; transform memakai angka itu pada data lain.
Pustaka adalah kumpulan fungsi siap pakai yang dipanggil dari kode.
Array adalah kumpulan nilai sejenis dengan bentuk yang tetap, yang dihitung sekaligus.
NumPy adalah pustaka untuk hitungan pada array.
Derau adalah gangguan acak pada jawaban.
Gauss pada derau ini adalah sebaran berbentuk lonceng: nilai kecil lebih sering daripada nilai besar.
Parameter noise pada make_regression adalah simpangan baku derau Gauss yang ditambahkan ke jawaban.
Parameter bias pada make_regression adalah geseran tetap, yaitu bilangan geser pada model linear yang membuat jawaban.
LinearRegression adalah regresi linear tanpa penalti pada fungsi kerugian.

Artinya: penalti yang ditambahkan pada fungsi kerugian regresi linear supaya koefisien tidak bebas membesar mengikuti kebetulan di data latih.
Artinya: regularisasi yang penalinya jumlah kuadrat koefisien. Koefisien mengecil ke arah nol, tetapi penalti kuadrat ini pada umumnya tidak membuat koefisien tepat nol.
Artinya: regularisasi yang penalinya jumlah nilai mutlak koefisien, dan penalti itu bisa membuat sebagian koefisien tepat nol.

Dua calon koefisien: A adalah 2 dan 2, B adalah 3 dan 0.

1. Berapa jumlah kuadrat A dan jumlah kuadrat B?
2. Berapa jumlah nilai mutlak A dan jumlah nilai mutlak B?

Hitung dengan tangan, lalu jalankan sel dari atas ke bawah.


In [2]:
# Satu ide: panggil NumPy.
import numpy as np


In [3]:
# Satu ide: simpan dua calon koefisien.
calon_a = np.array([2.0, 2.0])
calon_b = np.array([3.0, 0.0])
calon_a


array([2., 2.])

In [4]:
# Satu ide: penalti kuadrat, yaitu jumlah kuadrat koefisien.
kuadrat_a = float(np.sum(calon_a ** 2))
kuadrat_b = float(np.sum(calon_b ** 2))
print("tangan_kuadrat_a", kuadrat_a)
print("tangan_kuadrat_b", kuadrat_b)


tangan_kuadrat_a 8.0
tangan_kuadrat_b 9.0


In [5]:
# Satu ide: penalti nilai mutlak, yaitu jumlah nilai mutlak koefisien.
mutlak_a = float(np.sum(np.abs(calon_a)))
mutlak_b = float(np.sum(np.abs(calon_b)))
print("tangan_mutlak_a", mutlak_a)
print("tangan_mutlak_b", mutlak_b)


tangan_mutlak_a 4.0
tangan_mutlak_b 3.0


In [6]:
# Satu ide: penalti kuadrat lebih kecil pada A, penalti nilai mutlak lebih kecil pada B.
print("tangan_kuadrat_a_lebih_kecil", kuadrat_a < kuadrat_b)
print("tangan_mutlak_b_lebih_kecil", mutlak_b < mutlak_a)
assert kuadrat_a < kuadrat_b, "Penalti kuadrat calon A harus lebih kecil."
assert mutlak_b < mutlak_a, "Penalti nilai mutlak calon B harus lebih kecil."


tangan_kuadrat_a_lebih_kecil True
tangan_mutlak_b_lebih_kecil True


## Kasus

Data dibuat dengan `make_regression`, lalu kolom pertama dikalikan 50.
`n_samples=50`, `n_features=30`, `n_informative=5`, `noise=15.0`, `bias=2.5`.
`n_targets`, `effective_rank`, `tail_strength`, `shuffle`, dan `coef` tidak diisi.
Seperempat baris menjadi data uji.

DummyRegressor(strategy="mean") selalu menebak rata-rata jawaban data latih; strategy adalah pilihan cara baseline, dan mean berarti rata-rata. Nama fungsi itu bukan istilah baru.

alpha Ridge pada kasus adalah 5. alpha Lasso pada kasus adalah 0.5.


In [7]:
# Satu ide: panggil pembuat data sintetis.
from sklearn.datasets import make_regression


In [8]:
# Satu ide: buat baris jawaban dengan parameter yang dijelaskan di markdown.
X, y = make_regression(
    n_samples=50,
    n_features=30,
    n_informative=5,
    noise=15.0,
    bias=2.5,
    random_state=SEED,
)


In [9]:
# Satu ide: lebarkan kolom pertama supaya skalanya berbeda.
X = X.copy()
X[:, 0] = X[:, 0] * 50


In [10]:
# Satu ide: sisihkan seperempat baris sebagai data uji.
from sklearn.model_selection import train_test_split

indeks = np.arange(X.shape[0])
X_latih, X_uji, y_latih, y_uji, indeks_latih, indeks_uji = train_test_split(
    X, y, indeks, test_size=0.25, random_state=SEED
)
print("kasus_baris", int(X.shape[0]))
print("kasus_kolom", int(X.shape[1]))
print("kasus_baris_latih", int(len(y_latih)))
print("kasus_baris_uji", int(len(y_uji)))


kasus_baris 50
kasus_kolom 30
kasus_baris_latih 37
kasus_baris_uji 13


In [11]:
# Satu ide: panggil garis tanpa penalti, Ridge, Lasso, dan baseline.
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler


In [12]:
# Satu ide: baseline selalu menebak rata-rata jawaban data latih.
basis = DummyRegressor(strategy="mean")
basis.fit(X_latih, y_latih)
mae_basis = float(mean_absolute_error(y_uji, basis.predict(X_uji)))
print("kasus_mae_basis", mae_basis)


kasus_mae_basis 57.411810161225546


In [13]:
# Satu ide: regresi linear tanpa penalti, skala dihitung dari data latih.
skala_tanpa = StandardScaler().fit(X_latih)
model_tanpa = LinearRegression().fit(skala_tanpa.transform(X_latih), y_latih)
uji_tanpa = model_tanpa.predict(skala_tanpa.transform(X_uji))
latih_tanpa = model_tanpa.predict(skala_tanpa.transform(X_latih))
mae_uji_tanpa = float(mean_absolute_error(y_uji, uji_tanpa))
mae_latih_tanpa = float(mean_absolute_error(y_latih, latih_tanpa))
print("kasus_mae_uji_tanpa", mae_uji_tanpa)
print("kasus_mae_latih_tanpa", mae_latih_tanpa)


kasus_mae_uji_tanpa 37.180797267888494
kasus_mae_latih_tanpa 5.484318130333347


In [14]:
# Satu ide: latih Ridge alpha 5 dan ukur jaraknya.
skala_ridge = StandardScaler().fit(X_latih)
model_ridge = Ridge(alpha=5.0).fit(skala_ridge.transform(X_latih), y_latih)
koef_ridge = model_ridge.coef_
mae_uji_ridge = float(mean_absolute_error(y_uji, model_ridge.predict(skala_ridge.transform(X_uji))))
mae_latih_ridge = float(mean_absolute_error(y_latih, model_ridge.predict(skala_ridge.transform(X_latih))))
print("kasus_mae_uji_ridge", mae_uji_ridge)
print("kasus_mae_latih_ridge", mae_latih_ridge)
print("kasus_nol_ridge", int(np.sum(koef_ridge == 0.0)))
print("kasus_ridge_lebih_baik", mae_uji_ridge < mae_uji_tanpa)


kasus_mae_uji_ridge 15.56518243431388
kasus_mae_latih_ridge 14.36052852635922
kasus_nol_ridge 0
kasus_ridge_lebih_baik True


In [15]:
# Satu ide: Ridge alpha 5 lebih dekat pada data uji, dan koefisiennya tidak tepat nol.
assert float(model_ridge.alpha) == 5.0, "Alpha Ridge harus 5.0."
assert mae_uji_ridge < mae_uji_tanpa, (
    "Galat uji Ridge harus lebih kecil daripada regresi tanpa penalti."
)
assert int(np.sum(koef_ridge == 0.0)) == 0, "Koefisien Ridge tidak boleh tepat nol."


In [16]:
# Satu ide: Ridge dengan alpha 0 sama dengan regresi linear, dalam toleransi 1e-8.
skala_nol = StandardScaler().fit(X_latih)
model_nol = Ridge(alpha=0.0).fit(skala_nol.transform(X_latih), y_latih)
selisih_ridge_nol = float(
    np.max(np.abs(model_nol.predict(skala_nol.transform(X_uji)) - uji_tanpa))
)
assert selisih_ridge_nol < 1e-8, (
    "Ridge dengan alpha 0 harus sama dengan regresi linear dalam toleransi 1e-8."
)


In [17]:
# Satu ide: Lasso boleh membuat koefisien tepat nol.
skala_lasso = StandardScaler().fit(X_latih)
model_lasso = Lasso(alpha=0.5).fit(skala_lasso.transform(X_latih), y_latih)
koef_lasso = model_lasso.coef_
mae_uji_lasso = float(mean_absolute_error(y_uji, model_lasso.predict(skala_lasso.transform(X_uji))))
mae_latih_lasso = float(mean_absolute_error(y_latih, model_lasso.predict(skala_lasso.transform(X_latih))))
print("kasus_mae_uji_lasso", mae_uji_lasso)
print("kasus_mae_latih_lasso", mae_latih_lasso)
print("kasus_nol_lasso", int(np.sum(koef_lasso == 0.0)))
assert float(model_lasso.alpha) == 0.5, "Alpha Lasso harus 0.5."
assert np.any(koef_lasso == 0.0), "Lasso harus punya koefisien tepat nol."


kasus_mae_uji_lasso 14.731507595481556
kasus_mae_latih_lasso 8.010166534837767
kasus_nol_lasso 6


In [18]:
# Satu ide: rata-rata StandardScaler diambil dari data latih saja.
assert isinstance(skala_ridge, StandardScaler), (
    "Langkah skala harus StandardScaler."
)
rata_skala = skala_ridge.mean_
sama_skala = bool(np.allclose(rata_skala, X_latih.mean(axis=0)))
print("kasus_skala_latih", sama_skala)
assert np.allclose(rata_skala, X_latih.mean(axis=0)), (
    "Rata-rata StandardScaler harus sama dengan rata-rata data latih."
)


kasus_skala_latih True


In [19]:
# Satu ide: pemisahan yang sama berulang bila SEED sama.
_indeks_latih_lagi, indeks_uji_lagi = train_test_split(
    indeks, test_size=0.25, random_state=SEED
)
print("kasus_indeks_uji", indeks_uji.tolist())
print("kasus_indeks_uji_lagi", indeks_uji_lagi.tolist())
print("kasus_sama", indeks_uji.tolist() == indeks_uji_lagi.tolist())
assert indeks_uji.tolist() == indeks_uji_lagi.tolist(), (
    "Pemisahan dengan SEED yang sama harus memilih baris yang sama."
)


kasus_indeks_uji [13, 39, 30, 45, 17, 48, 26, 25, 32, 19, 12, 4, 37]
kasus_indeks_uji_lagi [13, 39, 30, 45, 17, 48, 26, 25, 32, 19, 12, 4, 37]
kasus_sama True
